## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. It also installs `scikit-learn`, which this analysis needs, into the same session. You can read every line of the code below before you run it.

# Loan defaults: two groups, debt-to-income and credit score

**Discriminant Analysis** · a worked example from the StatNex teaching guide

**The story.** A small lender looks back at 13 personal loans: 6 defaulted and 7 were repaid. For each it has the borrower's debt-to-income ratio, years in their job and credit score.

**The question.** Can debt-to-income and credit score tell future defaulters from good payers?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn"

## The table

This cell saves the table `09_multivariate/discriminant_analysis__2groups.csv` (13 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,Loan outcome,Debt-to-income (%),Years employed,Credit score
1.0,Defaulted,46.2,1.5,598.0
2.0,Defaulted,41.8,3.0,622.0
3.0,Defaulted,38.5,0.5,575.0
4.0,Defaulted,49.0,4.0,640.0
5.0,Defaulted,35.1,2.0,605.0
6.0,Defaulted,44.7,6.5,661.0
7.0,Repaid,31.4,5.0,688.0
8.0,Repaid,22.0,8.5,742.0
9.0,Repaid,36.9,2.5,701.0
10.0,Repaid,27.3,12.0,719.0
11.0,Repaid,25.8,1.0,655.0
12.0,Repaid,19.6,7.0,760.0
13.0,Repaid,29.9,3.5,694.0
""")

## Helper: collinear_columns

A step that no library does, written out. In the code download it is the file `collinear_columns.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

## Helper: discriminant

A step that no library does, written out. In the code download it is the file `discriminant.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""The arithmetic of canonical discriminant analysis that SciPy and statsmodels lack.

W is the within-group sums of squares and cross-products, B the between-group one.
Wilks' lambda = |W| / |W + B| is the share of the variables' variation the groups do
*not* explain; everything below is built from it.
"""

import numpy as np
from scipy import stats


def scatter_matrices(values, codes, groups):
    """W (within groups), B (between groups) and the grand mean."""
    grand = values.mean(axis=0)
    p = values.shape[1]
    within, between = np.zeros((p, p)), np.zeros((p, p))
    for k in range(groups):
        block = values[codes == k]
        centred = block - block.mean(axis=0)
        within += centred.T @ centred
        shift = (block.mean(axis=0) - grand)[:, None]
        between += len(block) * shift @ shift.T
    return within, between, grand


def wilks(values, codes, groups):
    """Wilks' lambda of the given variables (1 when there are none)."""
    if values.shape[1] == 0:
        return 1.0
    within, between, _ = scatter_matrices(values, codes, groups)
    return np.exp(np.linalg.slogdet(within)[1] - np.linalg.slogdet(within + between)[1])


def rao_f(wilks_lambda, p, g, n):
    """Rao's F approximation to Wilks' lambda: F, df1, df2, P (exact when min(p, g-1) <= 2)."""
    df1 = p * (g - 1)
    numerator, denominator = p**2 * (g - 1) ** 2 - 4, p**2 + (g - 1) ** 2 - 5
    s = np.sqrt(numerator / denominator) if numerator > 0 and denominator > 0 else 1.0
    df2 = (n - 1 - (p + g) / 2) * s - (df1 - 2) / 2
    root = wilks_lambda ** (1 / s)
    f = (1 - root) / root * df2 / df1
    return f, df1, df2, stats.f.sf(f, df1, df2)


def partial_f(full, reduced, g, n, variables):
    """Partial lambda (full / reduced) of one variable given the others, its F, df and P.

    `variables` counts the model that includes the variable.
    """
    partial = full / reduced
    df1, df2 = g - 1, n - g - variables + 1
    if df2 <= 0:
        return partial, np.nan, df1, max(df2, 0), np.nan
    f = (1 - partial) / partial * df2 / df1
    return partial, f, df1, df2, stats.f.sf(f, df1, df2)


def box_m(samples):
    """Box's M test of equal covariance matrices, in its chi-square form: M, chi2, df, P."""
    usable = [s for s in samples if s.shape[0] > s.shape[1]]
    p, k = usable[0].shape[1], len(usable)
    dfs = np.array([len(s) - 1 for s in usable], dtype=float)
    covariances = [np.cov(s, rowvar=False) for s in usable]
    pooled = sum(d * c for d, c in zip(dfs, covariances)) / dfs.sum()
    m = dfs.sum() * np.linalg.slogdet(pooled)[1] - sum(
        d * np.linalg.slogdet(c)[1] for d, c in zip(dfs, covariances))
    c1 = ((1 / dfs).sum() - 1 / dfs.sum()) * (2 * p * p + 3 * p - 1) / (6 * (p + 1) * (k - 1))
    df = (k - 1) * p * (p + 1) // 2
    return m, m * (1 - c1), df, stats.chi2.sf(m * (1 - c1), df)


def stepwise(values, codes, g, start, may_enter, may_remove, use_p, enter, remove):
    """Variable selection on partial Wilks' lambda; one record per step.

    Each step enters the best variable that qualifies (smallest P, or largest F); only
    when none does, it removes the worst that qualifies for removal (largest P, or
    smallest F). It stops when a step makes no move or would revisit a set.
    """
    n = len(codes)
    pool = range(values.shape[1])
    current, steps, seen = list(start), [], set()
    while tuple(current) not in seen:
        seen.add(tuple(current))
        base = wilks(values[:, current], codes, g)
        entered = removed = None
        if may_enter:
            best = None
            for j in pool:
                if j in current:
                    continue
                full = wilks(values[:, [*current, j]], codes, g)
                _, f, _, df2, p = partial_f(full, base, g, n, len(current) + 1)
                if df2 <= 0 or not np.isfinite(f):
                    continue
                qualifies, score = (p < enter, p) if use_p else (f > enter, -f)
                if qualifies and (best is None or score < best[0]):
                    best = (score, j)
            entered = best[1] if best else None
        if may_remove and entered is None and current:
            worst = None
            for j in current:
                reduced = wilks(values[:, [c for c in current if c != j]], codes, g)
                _, f, _, df2, p = partial_f(base, reduced, g, n, len(current))
                if df2 <= 0 or not np.isfinite(f):
                    continue
                qualifies, score = (p > remove, -p) if use_p else (f < remove, f)
                if qualifies and (worst is None or score < worst[0]):
                    worst = (score, j)
            removed = worst[1] if worst else None
        steps.append({"before": list(current), "entered": entered, "removed": removed})
        if entered is None and removed is None:
            break
        if entered is not None:
            current.append(entered)
        if removed is not None:
            current.remove(removed)
    return steps, current

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# collinear_columns: defined in the helper cell collinear_columns above (collinear_columns.py in the code download)
# box_m, partial_f, rao_f, scatter_matrices and wilks: defined in the helper cell discriminant above (discriminant.py in the code download)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# The response names each case's group; text in an explanatory column counts as missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Loan outcome"
predictors = ["Debt-to-income (%)", "Credit score"]
frame = data[predictors].apply(pd.to_numeric, errors="coerce")


# A group as text; a number read as 2.0 prints as "2".
def label(value):
    return str(int(value)) if isinstance(value, float) and value.is_integer() else str(value)

## Cases

In [ ]:
y_ok, x_ok = data[response].notna(), frame.notna().all(axis=1)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[y_ok & x_ok]
outcome = data.loc[y_ok & x_ok, response].map(label).to_numpy()
# Groups in the order they first appear, for now (they are reordered after the fit).
groups = list(dict.fromkeys(outcome))
codes = np.array([groups.index(v) for v in outcome])
values = valid.to_numpy(float)
n, g = len(codes), len(groups)

## Variables that are linear combinations of others

In [ ]:
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
all_predictors = list(predictors)
# They are left out of the analysis.
predictors = [name for name in predictors if name not in dropped]
values = valid[predictors].to_numpy(float)

## All variables

In [ ]:
chosen = list(range(len(predictors)))
names = [predictors[j] for j in chosen]
print("Variables in the functions:", names)

## The discriminant functions

In [ ]:
# scikit-learn's LinearDiscriminantAnalysis finds the weights that maximise
# between-group over within-group variation (the eigenvectors of W⁻¹B), at most
# min(p, g - 1) of them, each scaled so its scores have a pooled within-group variance
# of 1. They are signed here so their weights add up to a positive number: a function's
# sign is arbitrary, and this fixes it as StatNex does. transform() gives the scores,
# each case's value on each function measured from the overall means.
X = values[:, chosen]
within, between, grand = scatter_matrices(X, codes, g)
pooled = within / (n - g)
lda = LinearDiscriminantAnalysis(solver="svd").fit(X, codes)
# Convention: scikit-learn scales the scores so their within-group variance divides by
# n; MASS::lda, SPSS and StatNex divide by n - g, as a pooled variance does. Each
# function is rescaled to that (on 150 cases in 3 groups, by sqrt(147 / 150)).
spread = np.sqrt(np.diag(lda.scalings_.T @ pooled @ lda.scalings_))
signs = np.where(lda.scalings_.sum(axis=0) < 0, -1.0, 1.0)
vectors = lda.scalings_ / spread * signs
scores = lda.transform(X) / spread * signs
functions = [f"function {j + 1}" for j in range(vectors.shape[1])]
# Each function's eigenvalue is its between-group sum of squares over its within-group
# sum of squares: how many times the separation of the groups outweighs the scatter
# inside them. scikit-learn keeps only their shares, so they are worked out from the scores.
group_means = np.array([scores[codes == k].mean(axis=0) for k in range(g)])
between_ss = (np.bincount(codes, minlength=g)[:, None] * group_means**2).sum(axis=0)
within_ss = ((scores - group_means[codes]) ** 2).sum(axis=0)
eigenvalues = between_ss / within_ss

# The groups are listed by their centroid on the first function, largest first.
centroids = np.array([scores[codes == k].mean(axis=0) for k in range(g)])
order = list(np.argsort(-centroids[:, 0]))
groups = [groups[k] for k in order]
codes = np.array([order.index(c) for c in codes])
centroids = pd.DataFrame(centroids[order], index=groups, columns=functions)

## Groups, descriptive statistics and correlations (every variable given)

In [ ]:
response_summary = pd.DataFrame({"n": [*np.bincount(codes, minlength=g), n]}, index=[*groups, "total"])
response_summary["%"] = response_summary["n"] / n
parts = {"all": valid, **{name: valid[codes == k] for k, name in enumerate(groups)}}
descriptive = pd.concat({
    name: pd.DataFrame({"n": len(part), "mean": part[all_predictors].mean(),
                        "variance": part[all_predictors].var(ddof=1),
                        "SD": part[all_predictors].std(ddof=1), "min": part[all_predictors].min(),
                        "max": part[all_predictors].max()})
    for name, part in parts.items()
})
correlation = pd.concat({name: part[all_predictors].corr() for name, part in parts.items()})
print()
print(response_summary)
print()
print(descriptive)
print()
print(correlation)

## Significance of the selected variables

In [ ]:
# Wilks' lambda of the whole set, by Rao's F; then each variable's partial lambda.
k = len(chosen)
lam = wilks(X, codes, g)
selection_result = [k, 1 - lam, lam, *rao_f(lam, k, g, n)]
included = pd.DataFrame(
    [partial_f(lam, wilks(np.delete(X, j, axis=1), codes, g), g, n, k) for j in range(k)],
    index=names, columns=["lambda", "F", "df1", "df2", "P"])
print("\nvariables, eta2, lambda, F, df1, df2, P:", selection_result)
print(included)

# Eigenvalues: each function's share of the separation, and its correlation ratio.
eigen = pd.DataFrame({"function": np.arange(1, len(eigenvalues) + 1), "eigenvalue": eigenvalues,
                      "share": eigenvalues / eigenvalues.sum(),
                      "cumulative": np.cumsum(eigenvalues) / eigenvalues.sum(),
                      "eta2": eigenvalues / (1 + eigenvalues)})
# Bartlett's sequential test: is anything left from function j onwards?
function_tests = []
for j in range(len(eigenvalues)):
    remaining = np.prod(1 / (1 + eigenvalues[j:]))
    chi2 = -(n - 1 - (k + g) / 2) * np.log(remaining)
    df = (k - j) * (g - j - 1)
    function_tests.append([remaining, chi2, df, stats.chi2.sf(chi2, df)])
function_tests = pd.DataFrame(function_tests, columns=["lambda", "chi-square", "df", "P"])
print()
print(eigen.to_string(index=False))
print()
print(function_tests)

# Raw weights with the constant that centres the scores; standardised weights are the
# raw ones times each variable's pooled within-group SD.
coefficients = pd.DataFrame(np.vstack([vectors, -grand @ vectors]), index=[*names, "Constant"],
                            columns=functions)
standardised = pd.DataFrame(vectors * np.sqrt(np.diag(pooled))[:, None], index=names,
                            columns=functions)
print()
print(coefficients)
print()
print(standardised)

## Classification

In [ ]:
# Each case goes to the group whose mean is nearest in Mahalanobis distance (pooled
# within-group covariance): the discriminant rule once every group is given the same
# prior probability, so a large group does not attract the borderline cases. The
# squared distances are printed with the scores.
means = np.array([X[codes == k].mean(axis=0) for k in range(g)])
distances = cdist(X, means, metric="mahalanobis", VI=np.linalg.inv(pooled)) ** 2
equal_priors = LinearDiscriminantAnalysis(solver="svd", priors=np.full(g, 1 / g)).fit(X, codes)
predicted = equal_priors.predict(X)
classification = pd.crosstab(codes, predicted).reindex(index=range(g), columns=range(g), fill_value=0)
classification.index = classification.columns = groups
classification["hit rate"] = np.diag(classification) / classification.sum(axis=1)
hit_rate = (predicted == codes).mean()
print()
print(classification)
print(f"overall hit rate: {hit_rate:.4f}")
print()
print(centroids)

## What the result says

Discriminant analysis builds a score from the predictors that separates known groups as well as possible, here 6 defaulted loans and 7 repaid ones. Wilks' lambda is the share of the variation that the groups do not explain, so a small value is good. It is 0.20, with F = 19.4 and P = 0.0004. A P value this small means chance alone would rarely separate two groups this well, so the groups differ clearly.

The standardised discriminant coefficients, 0.66 for debt-to-income and -0.67 for credit score, show that the two predictors weigh about the same, in opposite directions. Defaulters carried more debt for their income (mean 42.6% against 27.6%) and had lower credit scores (617 against 708). The classification result puts all 13 loans in the right group.

But the rule was checked on the same 13 loans that built it. Whether it works as well on the next borrower is another matter.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.